Extracting the different datasets. Analysing the journals.

The Source Extraction, Works Extraction (BJP) and Works Extraction (all Pharmacology) are essentials. The Source Analysis is not the most interesting part and can be skipped.

In [18]:
import json 
import pyalex as alex
alex.config.email = "gabriel.vignon@ensta.fr"
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import aquarel
from IPython.display import clear_output

# Source Extraction

Extracting the data: 

In [ ]:
source = (
    alex.Sources()["s55107261"] # ID OpenAlex of the BJP
)

concurrence = ( 
    alex.Sources()
    .search("pharmacology OR pharmacological OR pharmaceutics OR pharmaceutical OR pharmacy OR drug OR drugs") # does not include some journals like Nature Reviews Drugs Discovery
) # 1672 found (info in concurrence.get().meta)


Saving the data:

In [15]:
with open("data/sources/bjp.json", "w") as f:
    json.dump(source, f, indent = 4)

book = []

# pages
for i in range(1, 10):
    book.append(concurrence.get(page = i, per_page = 200))

# Save the pages
for i in range(1, 10):
    with open(f"data/sources/concurrence_{i}.json", "w") as f:
        json.dump(book[i-1], f, indent = 4)

In [16]:
dfs_concurrence = []
for i in range(1, 10):
    with open(f"data/sources/concurrence_{i}.json", "r") as f:
        dfs_concurrence.append(pd.DataFrame.from_dict(json.load(f)))

df_concurrence = pd.concat(dfs_concurrence, ignore_index = True)
df_concurrence["impact"] = df_concurrence["cited_by_count"]/df_concurrence["works_count"]
df_concurrence = df_concurrence[df_concurrence["works_count"]>150]

# Source Analysis

### BJP only

In [ ]:
source_counts_by_year = source["counts_by_year"]

In [ ]:
with open("data/sources/bjp_counts_by_year.json", "w") as f:
    json.dump(source_counts_by_year, f, indent = 4)

In [ ]:
df_source = pd.read_json("data/sources/bjp_counts_by_year.json")
df_source.set_index("year", inplace = True)
df_source.drop(index = 2025, inplace = True) # dropping the current year
df_source.head()

In [ ]:
sns.lineplot(data = df_source, x = "year", y = "works_count")
plt.show()

In [ ]:
sns.lineplot(data = df_source, x = "year", y = "cited_by_count")

### BJP & concurrence

In [ ]:
top_20 = df_concurrence[["display_name", "impact", "cited_by_count", "host_organization", "works_count", "id"]].sort_values(by = "impact", ascending = False).reset_index()[:25]
# JIF = over the last 2 years
# Here: since the beginning  

#### Top journals

In [ ]:
with aquarel.load_theme("scientific"):
    plt.figure(figsize=(4.8,6.4))
    sns.barplot(data = top_20, x = "impact", y = "display_name", orient = "h", color="#3944f3", legend = True)
    #plt.xticks(rotation = 90)
    plt.xlabel("Global journal impact factor", fontsize=16)
    plt.ylabel("")
    plt.show()

In [ ]:
# df_concurrence_counts = pd.DataFrame({'year':[],'works_count':[],'cited_by_count':[]})

# for i in range(df_concurrence.shape[0]):
#     temp = df_concurrence.loc[i, "counts_by_year"]
#     for j in temp:
#         df_concurrence_counts = pd.concat([df_concurrence_counts, pd.DataFrame([j])], ignore_index = True)       

In [ ]:
df_avg_concurrence = df_concurrence_counts.groupby("year").mean()
df_avg_concurrence.drop(index = 2025, inplace = True)

In [ ]:
top = 0.9
toptop = 0.99

df_top_concurrence = df_concurrence_counts.groupby("year").quantile(top)
df_top_concurrence.drop(index = 2025, inplace = True)

df_toptop_concurrence = df_concurrence_counts.groupby("year").quantile(toptop)
df_toptop_concurrence.drop(index = 2025, inplace = True)

In [ ]:
df_source_and_co = df_source
df_source_and_co["works_count_avg_concurrence"] = df_avg_concurrence["works_count"]
df_source_and_co["cited_by_count_avg_concurrence"] = df_avg_concurrence["cited_by_count"]

df_source_and_co["works_count_top_concurrence"] = df_top_concurrence["works_count"]
df_source_and_co["cited_by_count_top_concurrence"] = df_top_concurrence["cited_by_count"]

df_source_and_co["works_count_toptop_concurrence"] = df_toptop_concurrence["works_count"]
df_source_and_co["cited_by_count_toptop_concurrence"] = df_toptop_concurrence["cited_by_count"]

In [ ]:
with aquarel.load_theme("scientific"):
    sns.lineplot(data = df_source_and_co, x = "year", y = "works_count", label = "BJP")
    sns.lineplot(data = df_source_and_co, x = "year", y = "works_count_avg_concurrence", label = "Sectorial mean")
    sns.lineplot(data = df_source_and_co, x = "year", y = "works_count_top_concurrence", label = f"Sectorial quantile {top}")
    sns.lineplot(data = df_source_and_co, x = "year", y = "works_count_toptop_concurrence", label = f"Sectorial quantile {toptop}")
    plt.legend(loc = "upper left")
    plt.show()

In [ ]:
with aquarel.load_theme("scientific"):
    sns.lineplot(data = df_source_and_co, x = "year", y = "cited_by_count", label = "BJP")
    sns.lineplot(data = df_source_and_co, x = "year", y = "cited_by_count_avg_concurrence", label = "Sectorial mean")
    sns.lineplot(data = df_source_and_co, x = "year", y = "cited_by_count_top_concurrence", label = f"Sectorial quartile {top}")
    sns.lineplot(data = df_source_and_co, x = "year", y = "cited_by_count_toptop_concurrence", label = f"Sectorial quartile {toptop}")
    plt.legend(loc = "upper left")
    plt.show()

## Analysis - Topic Share

In [ ]:
source_topic_share = source["topic_share"]
df_topic_share = pd.DataFrame(source_topic_share).sort_values(by = "value", ascending = False)

top5_topics_bjp = df_topic_share[["display_name", "value"]].head(n = 5)
top5_topics_bjp.columns = ["Topic", "Share (%)"]
top5_topics_bjp["Share (%)"] = top5_topics_bjp["Share (%)"] * 100
top5_topics_bjp.to_csv("data/sources/top5_topics_bjp.csv", index = False)

In [ ]:
topic_share_concurrence = pd.DataFrame()
for index in df_concurrence.index:
    for j in df_concurrence.loc[index, "topic_share"]:
        topic_share_concurrence = pd.concat([topic_share_concurrence, pd.DataFrame([j])], ignore_index = True)

In [ ]:
topic_share_concurrence_treated = (topic_share_concurrence[["display_name", "value"]].groupby(by = "display_name").mean()).sort_values(by = "value", ascending = False)

topic_share_concurrence_treated.reset_index(inplace = True)
top5_topics_concurrence = topic_share_concurrence_treated.loc[:,["display_name", "value"]].head(n = 5)
top5_topics_concurrence.columns = ["Topic", "Share (%)"]
top5_topics_concurrence["Share (%)"] = top5_topics_concurrence["Share (%)"] * 100
top5_topics_concurrence.to_csv("data/sources/top5_topics_concurrence.csv", index = False)

# Works Extraction (BJP)

### Latest version

In [ ]:
data = []

pager = alex.Works().filter(locations={"source": {"id": "s55107261"}}).paginate(per_page = 200, n_max = None)
for page in pager:
    for work in page:
        authors = []
        institutions = []
        countries = []
        for i in work.get("authorships"):
            authors.append((i.get("author")).get("id"))

            for j in i.get("institutions"):
                institutions.append([j.get("display_name"), j.get("id")])

            for j in i.get("countries"):
                countries.append(j)

        new = {
            "title": work.get("title"),
            "year": work.get("publication_year"),
            "cited_by_count": work.get("cited_by_count"),          
            "countries_distinct_count": work.get("countries_distinct_count"),
            "institutions_distinct_count": work.get("institutions_distinct_count"),
            "citation_normalized_percentile": work.get("citation_normalized_percentile"),
            "primary_topic": work.get("primary_topic"), 
            "keywords": work.get("keywords"), 
            "concepts": work.get("concepts"),
            "referenced_works_count": work.get("referenced_works_count"),
            "referenced_works": work.get("referenced_works"), 
            "abstract": work["abstract"],
            "abstract_inverted_index": work.get("abstract_inverted_index"),
            "journal":work.get("primary_location").get("source").get("display_name")
        }
        for i in range(1, len(authors) + 1):
            new[f"author_{i}"] = authors[i - 1]
        for i in range(1, len(institutions) + 1):
            new[f"institution_{i}"] = institutions[i - 1]
        for i in range(1, len(countries) + 1):
            new[f"country_{i}"] = countries[i - 1]
        for i in work.get("counts_by_year"):
            new[f"cited_by_count_{i.get("year")}"] = i.get("cited_by_count")
        if work.get("primary_topic") != None:
            new["primary_topic"] =  (work.get("primary_topic")).get("display_name"),
            new["primary_subfield"] =  ((work.get("primary_topic")).get("subfield")).get("display_name"),
            new["primary_field"] =  ((work.get("primary_topic")).get("field")).get("display_name"),
            new["primary_domain"] =  ((work.get("primary_topic")).get("domain")).get("display_name"),

        count = 1
        for i in work.get("keywords"):
            new[f"keyword_{count}"] = i.get("display_name")
            count += 1

        data.append(new)
data = pd.DataFrame(data)

In [ ]:
data.to_csv("data/works/works_journals.csv", index = False)

# Works Extraction (all pharmacology)

In [ ]:
top_50 = df_concurrence[["display_name", "impact", "cited_by_count", "host_organization", "works_count", "id"]].sort_values(by = "impact", ascending = False).reset_index()[:50]

In [ ]:
works_count = 0
for source in top_50["id"]:
    result = alex.Works().filter(locations={"source": {"id": source}}).get()
    works_count += result.meta["count"]

In [ ]:
data = []
progress = 0

for source in top_50["id"]:
    pager = alex.Works().filter(locations={"source": {"id": source}}).paginate(per_page = 200, n_max = None)
    for page in pager:
        for work in page:
            authors = []
            institutions = []
            countries = []
            for i in work.get("authorships"):
                authors.append((i.get("author")).get("id"))

                for j in i.get("institutions"):
                    institutions.append([j.get("display_name"), j.get("id")])

                for j in i.get("countries"):
                    countries.append(j)

            new = {
                "title": work.get("title"),
                "year": work.get("publication_year"),
                "cited_by_count": work.get("cited_by_count"),          
                "countries_distinct_count": work.get("countries_distinct_count"),
                "institutions_distinct_count": work.get("institutions_distinct_count"),
                "citation_normalized_percentile": work.get("citation_normalized_percentile"),
                "primary_topic": work.get("primary_topic"), 
                "keywords": work.get("keywords"), 
                "concepts": work.get("concepts"),
                "referenced_works_count": work.get("referenced_works_count"),
                "referenced_works": work.get("referenced_works"), 
                "abstract": work["abstract"],
                "abstract_inverted_index": work.get("abstract_inverted_index"),
                "journal":work.get("primary_location").get("source").get("display_name")

            }
            for i in range(1, len(authors) + 1):
                new[f"author_{i}"] = authors[i - 1]
            for i in range(1, len(institutions) + 1):
                new[f"institution_{i}"] = institutions[i - 1]
            for i in range(1, len(countries) + 1):
                new[f"country_{i}"] = countries[i - 1]
            for i in work.get("counts_by_year"):
                new[f"cited_by_count_{i.get("year")}"] = i.get("cited_by_count")
            if work.get("primary_topic") != None:
                new["primary_topic"] =  (work.get("primary_topic")).get("display_name"),
                new["primary_subfield"] =  ((work.get("primary_topic")).get("subfield")).get("display_name"),
                new["primary_field"] =  ((work.get("primary_topic")).get("field")).get("display_name"),
                new["primary_domain"] =  ((work.get("primary_topic")).get("domain")).get("display_name"),

            count = 1
            for i in work.get("keywords"):
                new[f"keyword_{count}"] = i.get("display_name")
                count += 1
            
            progress +=1
            if progress%1000 == 0:
                print(progress/works_count*100)
                clear_output(wait=True)

            data.append(new)

In [ ]:
data_chunks = [] # batching to avoid memory saturation
for i in range(0, len(data), 10_000):
    data_chunks.append(data[i:i + 10_000])

In [ ]:
data = pd.DataFrame()
for i in range(len(data_chunks)):
    data_chunks[i] = pd.DataFrame(data_chunks[i])
    

In [ ]:
df = pd.concat(data_chunks)

In [ ]:
df.to_csv("data_pharmacology/data_journals.csv")